# ML Deployment Tutorial

This notebook covers an end-to-end classification workflow using the real-world `breast_cancer` dataset from `scikit-learn`. You will: train a model, evaluate it, save the inference artifact, and prepare for deployment in a Flask app.

## 1. Problem statement

The goal is to predict whether a tumor is malignant or benign based on clinical measurements. This is a real healthcare classification problem used in many teaching and production scenarios.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set(style="whitegrid", palette="muted")

### Load the dataset and inspect the shape

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Dataset shape:", X.shape)
print("Target classes:", data.target_names)
X.head()

In [ ]:
class_counts = y.value_counts().rename({0: 'malignant', 1: 'benign'})
class_counts

## 2. Select features and split data

To keep deployment simple, this example uses five clinically meaningful features.

In [ ]:
feature_names = [
    'mean radius',
    'mean texture',
    'mean perimeter',
    'mean area',
    'mean smoothness'
]

X_selected = X[feature_names]
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.20, random_state=42, stratify=y
)

X_train.shape, X_test.shape

## 3. Scale the features and train the model

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy on test set: {accuracy:.4f}')

In [ ]:
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=data.target_names, yticklabels=data.target_names)
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.title("Confusion Matrix")
plt.show()

## 4. Save the model artifact for deployment

The Flask application will load this saved `model.pkl` file for inference.

In [ ]:
model_package = {
    'model': model,
    'scaler': scaler,
    'feature_names': feature_names,
    'target_names': list(data.target_names),
}
joblib.dump(model_package, 'model.pkl')
print('Saved model artifact to model.pkl')

## 5. Deployment notes

### Local deployment
1. Install dependencies: `pip install -r requirements.txt`
2. Run the Flask app: `python app.py`
3. Open `http://localhost:5000` in your browser.

### AWS EC2 deployment
- Launch an EC2 instance with Python installed.
- Transfer the `Deployment` folder to the instance.
- Create and activate a virtual environment on EC2.
- Install dependencies and run `python app.py`.
- Open the EC2 public IP and port `5000` in your browser.

### Why this notebook helps teaching
- It shows every step of the ML pipeline.
- It saves the exact artifact the Flask app uses.
- It keeps the model and scaler together so inference is correct.